# 01 — LSTM univariado: pH da estação EF01 (CETESB)

**Objetivo:** primeiro modelo neural no pH, no mesmo desenho do `00-baseline-ph` (L=8640/H=288, split 70/15/15 sem shuffle + holdout puro de 10 dias, 10 origens diárias). A régua a bater é o **sazonal-naive: MAE 0,0501 (rolante) / 0,0466 (holdout diário)**.
**Decisão de custo (documentada):** LSTM de 8640 passos a 5 min é inviável em CPU no tempo-alvo (5–10 min). Como o ARIMA no 00, o LSTM roda em **grade horária** (`Lh=720h`, `Hh=24h`, média horária) e cada previsão horária é repetida 12× para voltar aos 5 min. As **janelas, splits, alvos e métricas são os mesmos do 00** — só a representação de entrada muda (para menos informação, nunca para mais).
**Dados:** `dados/ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`. Saída direta multi-step (sem rollout).

In [1]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "01-lstm-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- protocolo travado (igual ao 00) ---
L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
PROVISORIO_CORTE = "2026-08-22 09:00"
HOLDOUT_DIAS = 10

# --- LSTM em grade horária (aproximação de custo, cf. ARIMA no 00) ---
LH, HH = 720, 24
HIDDEN, LAYERS, DROPOUT = 32, 1, 0.0
BATCH, LR = 256, 1e-3
MAX_EPOCHS, PATIENCE = 50, 8
TRAIN_STRIDE, VAL_STRIDE = 8, 8
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)


ROOT: /home/marcos/Projetos/temporal-model | CSV existe: True | torch: 2.14.0+cpu


## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()


(26209, 2) 2026-06-01 00:00:00 → 2026-08-31 00:00:00
faltantes: 4794 (18.3%)


,ds,y
count,26209,21415.000000
mean,2026-07-16 12:00:00,6.142466
min,2026-06-01 00:00:00,5.560000
25%,2026-06-23 18:00:00,5.970000
50%,2026-07-16 12:00:00,6.170000
75%,2026-08-08 06:00:00,6.280000
max,2026-08-31 00:00:00,6.590000
std,NaN,0.198893


## 2. EDA — perfil, faltantes e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvline(pd.Timestamp(PROVISORIO_CORTE), color="r", ls="--", lw=1)
ax[0].set_title("pH EF01 — série completa (vermelho = início do trecho provisório)")
ax[0].set_ylabel("pH")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do pH")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("pH por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")


maior gap: 18 passos = 1.5 h | gaps > 24 passos: 0


fig salva: /home/marcos/Projetos/temporal-model/resultados/01-lstm-ph/figs/01-eda.png


## 3. Limpeza — grade completa + interpolação limitada
Reindex na grade de 5 min, flag do trecho provisório e interpolação temporal de no máximo 2 h (igual ao 00).

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
provisorio = s.index > pd.Timestamp(PROVISORIO_CORTE)
print(f"trecho provisório: {int(provisorio.sum())} slots ({100*provisorio.mean():.1f}%)")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")


slots na grade: 26209 | linhas no CSV: 26209
NaN após interpolação (limite 24): 0
trecho provisório: 2484 slots (9.5%)


fig salva


## 4. Estacionariedade (ADF) e decomposição STL
Idêntico ao 00 (últimos 4032 pontos do treino, período 288).

In [5]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")


ADF stat=-4.66 p-valor=0.000102 → estacionária


fig salva


## 5. Janelamento + holdout puro
Amostras `(L=8640 → H=288)` por janela deslizante, só janelas 100% observadas. Pré-holdout: split 70/15/15 **sem shuffle**. Holdout: últimos 10 dias + 10 origens diárias. **Idêntico ao 00** — o LSTM será avaliado nestas mesmas janelas.

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
splits = {"train": tr, "val": va, "test": te, "holdout": ho}
for k, idx in splits.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
print(f"zona holdout (alvos): {ZONE.date()} → {s.index.max().date()}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TR_END = ends[tr[-1]]


train: 10281 janelas | alvos 2026-07-01 → 2026-08-06
val: 2203 janelas | alvos 2026-08-06 → 2026-08-14
test: 2204 janelas | alvos 2026-08-14 → 2026-08-21
holdout: 2594 janelas | alvos 2026-08-21 → 2026-08-31
janelas descartadas (com NaN): 0
zona holdout (alvos): 2026-08-21 → 2026-08-31
dias previstos: ['2026-08-21', '2026-08-22', '2026-08-23', '2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-29', '2026-08-30']


## 6. Baselines baratos (teste rolante + holdout)
Persistência, sazonal-naive (lag 288) e média móvel 288 — vetorizados, mesmos do 00.

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xte, Yte = X[te], Y[te]
Xho, Yho = X[ho], Y[ho]
pred_te = cheap_preds(Xte)
pred_ho = cheap_preds(Xho)
print("teste rolante:")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())


teste rolante:
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0727  0.0942  1.1683  1.1657
sazonal_naive_288  0.0501  0.0648  0.8079  0.8051
media_movel_288    0.0631  0.0768  1.0159  1.0133


## 7. LSTM em grade horária — treino
Série horária `hs` (média de 1 h). Mapeamento **sem vazamento**: `t0` = 1º timestamp do alvo (5 min), `a = floor(t0, 1h)`; contexto `hs[a-720h:a-1h]` (720) e alvo `hs[a:a+23h]` (24) — o contexto termina 1 h antes do dia previsto começar. Normalização z-score fitada **só até `TR_END`**. Subamostra do treino/val por stride (custo) — **avaliação (§8–§9) usa todas as origens**. Early stopping na val (MSE horária normalizada).

In [8]:
hs = s.resample("1h").mean()
print(f"hs: {len(hs)} horas | NaN: {int(hs.isna().sum())} | {hs.index.min()} → {hs.index.max()}")
MU = float(hs.loc[:TR_END].mean())
SIG = float(hs.loc[:TR_END].std())
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "modelos" / "normalizacao.json").write_text(json.dumps({"mu": MU, "sigma": SIG, "ate": str(TR_END)}))
print(f"z-score: mu={MU:.4f} sigma={SIG:.4f} (fit até {TR_END.date()})")

def ctx_tgt(e):
    raise RuntimeError("removido: ver mapeamento vetorizado abaixo (sem vazamento)")

# --- mapeamento vetorizado origem-5min -> posição horária (sem vazamento) ---
from numpy.lib.stride_tricks import sliding_window_view as _swv
hs_vals = hs.to_numpy().astype(np.float32)
Wctx = _swv(hs_vals, LH)
Wtgt = _swv(hs_vals, HH)
t0_all = (ends - pd.Timedelta(minutes=5 * (H - 1))).floor("h")
pos_a = hs.index.get_indexer(t0_all)
valid_all = (pos_a >= LH) & (pos_a + HH <= len(hs_vals))
print(f"janelas com contexto/alvo horário válidos: {int(valid_all.sum())}/{len(ends)}")

# monta tensores (treino/val com stride; teste/holdout/diário completos na §8)
def monta(idxs):
    ii = np.asarray(idxs)[valid_all[np.asarray(idxs)]]
    Xh = ((Wctx[pos_a[ii] - LH] - MU) / SIG).astype(np.float32)
    Yh = ((Wtgt[pos_a[ii]] - MU) / SIG).astype(np.float32)
    return Xh, Yh, ii

Xtr_h, Ytr_h, keep_tr = monta(tr[::TRAIN_STRIDE])
Xva_h, Yva_h, keep_va = monta(va[::VAL_STRIDE])
print(f"treino-h: {Xtr_h.shape} (stride {TRAIN_STRIDE}, {len(keep_tr)}/{len(tr)}) | val-h: {Xva_h.shape} (stride {VAL_STRIDE})")

class LSTMForecaster(nn.Module):
    def __init__(self, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT, h_out=HH):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, h_out)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

model = LSTMForecaster().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()
tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_h[..., None]), torch.from_numpy(Ytr_h)), batch_size=BATCH, shuffle=True)
va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_h[..., None]), torch.from_numpy(Yva_h)), batch_size=512)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params}")

best, patience, hist = float("inf"), 0, {"train": [], "val": []}
t0 = time.time()
for ep in range(1, MAX_EPOCHS + 1):
    model.train()
    tl = 0.0
    for xb, yb in tr_loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        tl += float(loss.detach()) * len(xb)
    tl /= len(tr_loader.dataset)
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for xb, yb in va_loader:
            vl += float(loss_fn(model(xb), yb)) * len(xb)
    vl /= len(va_loader.dataset)
    hist["train"].append(tl); hist["val"].append(vl)
    tag = ""
    if vl < best:
        best, patience = vl, 0
        torch.save({"state": model.state_dict(), "cfg": {"hidden": HIDDEN, "layers": LAYERS, "dropout": DROPOUT, "lh": LH, "hh": HH}, "norm": {"mu": MU, "sigma": SIG}}, OUT / "modelos" / "lstm_ph.pt")
        tag = " *"
    else:
        patience += 1
    print(f"ep {ep:02d} train={tl:.4f} val={vl:.4f}{tag}", flush=True)
    if patience >= PATIENCE:
        print(f"early stopping na ep {ep} (best val={best:.4f})")
        break
print(f"treino em {time.time()-t0:.0f}s | melhor val={best:.4f} | modelo: modelos/lstm_ph.pt")

ckpt = torch.load(OUT / "modelos" / "lstm_ph.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["state"])
model.eval()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(hist["train"], label="treino")
ax.plot(hist["val"], label="val")
ax.set_title("LSTM-h — loss por época (MSE horária normalizada)")
ax.set_xlabel("época"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png")
print("fig salva: 07-curvas-treino.png")


hs: 2185 horas | NaN: 0 | 2026-06-01 00:00:00 → 2026-08-31 00:00:00
z-score: mu=6.0582 sigma=0.1999 (fit até 2026-08-06)
janelas com contexto/alvo horário válidos: 17282/17282
treino-h: (1286, 720) (stride 8, 1286/10281) | val-h: (276, 720) (stride 8)


params: 5272


ep 01 train=1.4675 val=1.6012 *


ep 02 train=1.4217 val=1.5243 *


ep 03 train=1.3737 val=1.4400 *


...

ep 29 train=0.2802 val=0.2144


ep 30 train=0.2739 val=0.2142


ep 31 train=0.2630 val=0.2168


ep 32 train=0.2569 val=0.2177


early stopping na ep 32 (best val=0.2119)
treino em 265s | melhor val=0.2119 | modelo: modelos/lstm_ph.pt
fig salva: 07-curvas-treino.png


## 8. Avaliação do LSTM nas janelas do protocolo
Inferência em **todas** as origens do teste rolante, do holdout e do holdout diário. Previsão horária (24) → `repeat ×12` → 288 passos de 5 min, comparada ao alvo `Y` de 5 min (mesma aproximação do ARIMA no 00).

In [9]:
def expand12(fc_h):
    return np.repeat(np.asarray(fc_h), 12, axis=1)[:, :H]

@torch.no_grad()
def prevê_h(idxs, batch=512):
    Xh, _, ii = monta(idxs)
    print(f"  {len(ii)}/{len(np.asarray(idxs))} origens válidas")
    Xt = torch.from_numpy(Xh[..., None])
    outs = []
    for b in range(0, len(Xt), batch):
        outs.append(model(Xt[b:b+batch]).numpy())
    return (np.concatenate(outs) * SIG + MU), ii

t0 = time.time()
Fte_h, _ = prevê_h(te)
Fho_h, _ = prevê_h(ho)
Fd_h, _ = prevê_h(daily_idx)
Pl_te, Pl_ho, Pl_d = expand12(Fte_h), expand12(Fho_h), expand12(Fd_h)
print(f"inferência em {time.time()-t0:.0f}s | teste {Pl_te.shape} holdout {Pl_ho.shape} diário {Pl_d.shape}")
print("LSTM teste rolante:", {k: round(v, 4) for k, v in metricas(Yte, Pl_te).items()})
print("LSTM holdout diário:", {k: round(v, 4) for k, v in metricas(Y[daily_idx], Pl_d).items()})


  2204/2204 origens válidas


  2594/2594 origens válidas


  10/10 origens válidas
inferência em 2s | teste (2204, 288) holdout (2594, 288) diário (10, 288)
LSTM teste rolante: {'MAE': 0.112, 'RMSE': 0.1255, 'MAPE': 1.8126, 'sMAPE': 1.7943}
LSTM holdout diário: {'MAE': 0.1081, 'RMSE': 0.1269, 'MAPE': 1.7518, 'sMAPE': 1.7315}


## 9. Comparação final + holdout dia a dia
Tabela do teste rolante (todas as origens), tabela do holdout diário (10 dias) e MAE por dia. Réguas do 00 impressas para referência.

In [10]:
linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
linhas["lstm_h"] = metricas(Yte, Pl_te)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante ===")
print(tab.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_te}
diario["lstm_h"] = metricas(Yd, Pl_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário (10 dias) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_te},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["lstm_h"] = [mae(Yd[k:k+1], Pl_d[k:k+1]) for k in range(len(Yd))]
print(por_dia.round(4).to_string())
print(f"\nRégua 00 (teste rolante): sazonal_naive_288 = 0.0501 | este exp: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua 00 (holdout diário): sazonal_naive_288 = 0.0466 | este exp: {tab_d['MAE'].idxmin()} = {tab_d['MAE'].min():.4f}")


=== teste rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0727  0.0942  1.1683  1.1657
sazonal_naive_288  0.0501  0.0648  0.8079  0.8051
media_movel_288    0.0631  0.0768  1.0159  1.0133
lstm_h             0.1120  0.1255  1.8126  1.7943
=== holdout diário (10 dias) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0973  0.1204  1.5774  1.5599
sazonal_naive_288  0.0466  0.0673  0.7501  0.7497
media_movel_288    0.0655  0.0823  1.0547  1.0540
lstm_h             0.1081  0.1269  1.7518  1.7315
            persistencia  sazonal_naive_288  media_movel_288  lstm_h
2026-08-21        0.0642             0.0317           0.0562  0.1072
2026-08-22        0.0686             0.0363           0.0485  0.1043
2026-08-23        0.0725             0.0444           0.0631  0.0878
2026-08-24        0.0972             0.0414           0.0508  0.0954
2026-08-25        0.0589             0.0380           0.0498  0.1054
2026-08-26        0.1066       

In [11]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_te["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, Pl_te[k], lw=1, alpha=0.9, label="lstm-h (×12)")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante — baselines + LSTM-h (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yd))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yd[k], "k-", lw=1.2, label="real")
    ax.plot(tf, cheap_preds(X[daily_idx])["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, cheap_preds(X[daily_idx])["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Pl_d[k], lw=1, alpha=0.9, label="lstm-h")
    ax.set_title(f"dia previsto {ends[daily_idx[k]].date()} (MAE lstm={por_dia['lstm_h'].iloc[k]:.3f} vs saz={por_dia['sazonal_naive_288'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")
print("figs salvas")


figs salvas


## 10. Conclusões e próximos passos

- A régua do 00 (sazonal-naive 0,0501 / 0,0466) está impressa na §9 para comparação direta.
- O LSTM-h opera em grade horária com expansão ×12 (aproximação de custo documentada na abertura); conte-o como "primeiro neural" e julgue pelo MAE nas mesmas janelas.
- Se o LSTM-h não bater o sazonal-naive, os candidatos seguintes são: (i) LSTM com mais contexto/resolução, (ii) DLinear/LightGBM barato (tese Zeng §3.2), (iii) PatchTST (§3.3).
- Artefatos em `resultados/01-lstm-ph/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `modelos/lstm_ph.pt`, `modelos/normalizacao.json` e `figs/` (inclui `07-curvas-treino.png`).